# weight-decay-decoupled — worked example 2: Show AdamW and L2-Adam diverge after one step

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

AdamW (decoupled) and Adam+L2 (coupled) produce identical updates only when the gradient scale matches the adaptive normalization. In practice they differ because Adam divides by `sqrt(v_hat)`, which rescales the effective step size per-coordinate. When the gradient variance is non-uniform, L2 folded into the gradient is rescaled differently than the decoupled direct decay, causing the two trajectories to diverge.

## Worked solution

**Step 1 — set up two separate trajectories from the same starting point.**
Both paths start with identical `p`, `grad`, `m=0`, `v=0`. We clone `p` so mutations to one path don't affect the other.

**Step 2 — AdamW path (decoupled).**
Apply `p_aw = p * (1 - lr*wd)` first, then standard Adam with the raw gradient. The decay term appears nowhere in the moment update.

**Step 3 — Adam+L2 path (coupled).**
Add `wd * p` to the gradient before the moment update: `g_eff = grad + wd * p`. Then run standard Adam on `g_eff`. The weight decay is now baked into the gradient, and the adaptive `sqrt(v_hat)` will normalize it differently.

**Step 4 — observe the divergence.**
Print both results. With non-trivial `wd` and non-zero initial parameter, the two should give slightly different values, demonstrating that L2 and AdamW are not interchangeable.

In [ ]:
import torch as t

t.manual_seed(5)
p_init = t.tensor([0.8])
grad = t.tensor([0.2])
lr, beta1, beta2, eps, wd = 5e-3, 0.9, 0.999, 1e-8, 0.1
step = 1

# AdamW path (decoupled)
p_aw = p_init.clone()
m_aw = t.zeros_like(p_aw)
v_aw = t.zeros_like(p_aw)
p_aw.mul_(1 - lr * wd)                         # decay first
m_aw = beta1 * m_aw + (1 - beta1) * grad
v_aw = beta2 * v_aw + (1 - beta2) * grad**2
m_hat_aw = m_aw / (1 - beta1**step)
v_hat_aw = v_aw / (1 - beta2**step)
p_aw.addcdiv_(m_hat_aw, v_hat_aw.sqrt().add_(eps), value=-lr)

# Adam+L2 path (coupled)
p_l2 = p_init.clone()
m_l2 = t.zeros_like(p_l2)
v_l2 = t.zeros_like(p_l2)
g_eff = grad + wd * p_l2                        # fold wd into gradient
m_l2 = beta1 * m_l2 + (1 - beta1) * g_eff
v_l2 = beta2 * v_l2 + (1 - beta2) * g_eff**2
m_hat_l2 = m_l2 / (1 - beta1**step)
v_hat_l2 = v_l2 / (1 - beta2**step)
p_l2.addcdiv_(m_hat_l2, v_hat_l2.sqrt().add_(eps), value=-lr)

print('AdamW (decoupled) p:', p_aw.item())
print('Adam+L2 (coupled) p:', p_l2.item())
print('Paths diverge:', not t.allclose(p_aw, p_l2))